# Линейные модели с нуля: сравнение с sklearn

Ноутбук проверяет собственные `LinearRegression` и `LogisticRegression`
(пакет `linear_models`) на реальных и синтетических данных. Каждая модель
обучается и проверяется на **тех же данных**, что и её аналог из `sklearn`.

Разделы:
1. Данные
2. Линейная регрессия: `analytic` и `gradient` против `LinearRegression` и `Ridge`
3. Логистическая регрессия против `sklearn.linear_model.LogisticRegression`
4. Режимы: без штрафа, L1, L2, с `intercept` и без
5. Сходимость: `loss_history_` и `learning_rate`
6. Выводы и объяснение расхождений

In [ ]:
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.linear_model import LinearRegression as SkLinearRegression
from sklearn.linear_model import LogisticRegression as SkLogisticRegression
from sklearn.linear_model import Ridge as SkRidge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from linear_models.linear_regression import LinearRegression
from linear_models.logistic_regression import LogisticRegression

SEED = 42
np.set_printoptions(precision=6, suppress=True)

print("python      :", sys.version.split()[0])
print("numpy       :", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("matplotlib  :", matplotlib.__version__)
print("свои классы :", LinearRegression.__name__, LogisticRegression.__name__)
print("sklearn     :", SkLinearRegression.__name__, SkRidge.__name__, SkLogisticRegression.__name__)

## 1. Данные

Четыре набора: для регрессии синтетика с известными весами и `load_diabetes`,
для классификации синтетика и `load_breast_cancer`.

- Одно разбиение train/test на набор: `test_size=0.25`, `random_state=SEED`
  (для классификации со `stratify=y`, чтобы доли классов совпадали).
- Признаки стандартизируются: **среднее и стандартное отклонение считаются
  только по train**, к test применяется то же преобразование.
- Наши модели и модели `sklearn` получают **одни и те же** массивы.
- Масштабирование нужно из-за градиентного спуска с фиксированным
  `learning_rate`: на сырых признаках `load_breast_cancer` он не успевает
  сойтись (это показано в разделе про сходимость).

In [ ]:
TEST_SIZE = 0.25


def split_and_scale(X, y, stratify=None):
    """Одно разбиение train/test и стандартизация по train."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED, stratify=stratify
    )
    scaler = StandardScaler().fit(X_train)
    return {
        "X_train": scaler.transform(X_train),
        "X_test": scaler.transform(X_test),
        "y_train": y_train,
        "y_test": y_test,
    }


rng = np.random.default_rng(SEED)
n, d = 300, 3
W_TRUE = np.array([2.0, -1.0, 0.5])
B_TRUE = 1.0
X_syn = rng.standard_normal((n, d))
z_syn = X_syn @ W_TRUE + B_TRUE

y_syn_reg = z_syn + 0.5 * rng.standard_normal(n)
y_syn_clf = (rng.random(n) < 1 / (1 + np.exp(-z_syn))).astype(int)

X_dia, y_dia = load_diabetes(return_X_y=True)
X_bc, y_bc = load_breast_cancer(return_X_y=True)

REG_DATA = {
    "synthetic": split_and_scale(X_syn, y_syn_reg),
    "diabetes": split_and_scale(X_dia, y_dia),
}
CLF_DATA = {
    "synthetic": split_and_scale(X_syn, y_syn_clf, stratify=y_syn_clf),
    "breast_cancer": split_and_scale(X_bc, y_bc, stratify=y_bc),
}

for group in (REG_DATA, CLF_DATA):
    for name, s in group.items():
        assert s["X_train"].shape[0] == s["y_train"].shape[0]
        assert s["X_test"].shape[0] == s["y_test"].shape[0]
        assert np.allclose(s["X_train"].mean(axis=0), 0.0, atol=1e-10)
        assert np.allclose(s["X_train"].std(axis=0), 1.0)

for title, group in (("регрессия", REG_DATA), ("классификация", CLF_DATA)):
    print(title)
    for name, s in group.items():
        print(
            f"  {name:14s} train {s['X_train'].shape}, test {s['X_test'].shape}",
            f"| y_train mean {s['y_train'].mean():.4f}, y_test mean {s['y_test'].mean():.4f}",
        )
print("W_TRUE =", W_TRUE, " B_TRUE =", B_TRUE)
print("classes in synthetic y:", np.bincount(y_syn_clf))
print("synthetic X_train mean:", CLF_DATA["synthetic"]["X_train"].mean(axis=0))

## 2. Линейная регрессия

### 2.1. `solver="analytic"` без штрафа против `sklearn.linear_model.LinearRegression`

Обе модели решают одну задачу наименьших квадратов точным методом, поэтому
ожидается совпадение до ошибок округления (порядка `1e-13` и меньше), а не
просто «разумная близость». Сравниваются коэффициенты, `intercept`,
предсказания на test и метрики на test (MSE и R²), посчитанные одной и той же
функцией `sklearn.metrics` для обеих моделей.

In [ ]:
def max_abs_diff(a, b):
    """Наибольшее по модулю отличие двух массивов."""
    return float(np.max(np.abs(np.asarray(a) - np.asarray(b))))


def report_regression(title, own, sk, data):
    """Обучает обе модели на train и печатает сравнение на одних и тех же данных."""
    own.fit(data["X_train"], data["y_train"])
    sk.fit(data["X_train"], data["y_train"])
    pred_own = own.predict(data["X_test"])
    pred_sk = sk.predict(data["X_test"])
    y_test = data["y_test"]
    print(title)
    print(f"  макс |разность коэффициентов| : {max_abs_diff(own.coef_, sk.coef_):.2e}")
    print(f"  |разность intercept|          : {abs(own.intercept_ - sk.intercept_):.2e}")
    print(f"  макс |разность предсказаний|  : {max_abs_diff(pred_own, pred_sk):.2e}")
    print(f"  test MSE  свой / sklearn      : {mean_squared_error(y_test, pred_own):.6f} / {mean_squared_error(y_test, pred_sk):.6f}")
    print(f"  test R2   свой / sklearn      : {r2_score(y_test, pred_own):.6f} / {r2_score(y_test, pred_sk):.6f}")
    return own, sk

In [ ]:
for name, data in REG_DATA.items():
    own, sk = report_regression(
        f"{name}: analytic vs LinearRegression",
        LinearRegression(solver="analytic"),
        SkLinearRegression(),
        data,
    )
    assert max_abs_diff(own.coef_, sk.coef_) < 1e-8
    assert abs(own.intercept_ - sk.intercept_) < 1e-8
    if name == "synthetic":
        print("  свои coef_:", own.coef_, " intercept_:", round(own.intercept_, 6), " (истина", W_TRUE, B_TRUE, ")")

### 2.2. `solver="gradient"` без штрафа против `sklearn.linear_model.LinearRegression`

Параметры спуска: `learning_rate=0.4`, `max_iter=20000`, `tol=0.0`.

- `learning_rate=0.4` меньше границы устойчивости `2 / λ_max` (для `diabetes`
  она равна `0.510`, для синтетики `1.823`), где `λ_max` это наибольшее
  собственное число матрицы `XᵀX / n`.
- `tol=0.0` выключает раннюю остановку, чтобы сравнивать именно точность
  оптимизации, а не критерий остановки (его разберём в разделе про сходимость).
- `max_iter=20000` выбрано по замеру: на `diabetes` самое медленное направление
  затухает с множителем около `1 − 0.4 · 0.0094 ≈ 0.9962` за шаг, этого хватает
  для точности порядка `1e-12`.

Кроме сравнения с `sklearn` печатается значение цели `J` у `analytic` и
`gradient`: у точного решения и у сошедшегося спуска оно должно совпасть.

In [ ]:
GRAD_PARAMS = {"solver": "gradient", "learning_rate": 0.4, "max_iter": 20000, "tol": 0.0}

for name, data in REG_DATA.items():
    own, sk = report_regression(
        f"{name}: gradient vs LinearRegression",
        LinearRegression(**GRAD_PARAMS),
        SkLinearRegression(),
        data,
    )
    ana = LinearRegression(solver="analytic").fit(data["X_train"], data["y_train"])
    print(f"  n_iter_: {own.n_iter_}")
    print(f"  J (цель) analytic / gradient  : {ana.loss_history_[-1]:.6f} / {own.loss_history_[-1]:.6f}")
    assert own.n_iter_ == GRAD_PARAMS["max_iter"]
    assert max_abs_diff(own.coef_, sk.coef_) < 1e-8
    assert abs(own.intercept_ - sk.intercept_) < 1e-8

### 2.3. L2: `analytic` и `gradient` против `sklearn.linear_model.Ridge`

Цели записаны по-разному. Наша: `(1/2n) · Σ (zᵢ − yᵢ)² + alpha · ½ ‖w‖²`.
У `Ridge`: `Σ (zᵢ − yᵢ)² + α_sk · ‖w‖²`. Если умножить нашу цель на `2n`,
получится `Σ (zᵢ − yᵢ)² + n · alpha · ‖w‖²`, поэтому **`α_sk = n · alpha`**,
где `n` это число объектов в train. Свободный член не штрафуется ни у нас, ни у
`Ridge`.

В таблице: наибольшая по модулю разность коэффициентов, `intercept` и предсказаний
на test, затем MSE на test для нашей модели и `sklearn`. Последняя строка
показывает, что бывает, если забыть множитель `n`.

Для L2 граница устойчивости спуска сдвигается: `learning_rate < 2 / (λ_max + alpha)`.
Для `diabetes` при `alpha = 0.5` она равна `0.4526`, наш `learning_rate = 0.4` ниже.

In [ ]:
def regression_row(own, sk, data):
    """Обучает обе модели на train и возвращает числа для одной строки сравнения."""
    own.fit(data["X_train"], data["y_train"])
    sk.fit(data["X_train"], data["y_train"])
    pred_own = own.predict(data["X_test"])
    pred_sk = sk.predict(data["X_test"])
    y_test = data["y_test"]
    return {
        "coef": max_abs_diff(own.coef_, sk.coef_),
        "intercept": abs(own.intercept_ - sk.intercept_),
        "pred": max_abs_diff(pred_own, pred_sk),
        "mse_own": mean_squared_error(y_test, pred_own),
        "mse_sk": mean_squared_error(y_test, pred_sk),
    }

In [ ]:
ALPHAS = (0.1, 0.5)

print(f"{'набор':10s} {'alpha':>5s} {'solver':>9s} {'coef':>9s} {'intercept':>9s} {'pred':>9s}   test MSE свой / sklearn")
for name, data in REG_DATA.items():
    n_train = data["X_train"].shape[0]
    for alpha in ALPHAS:
        for solver_params in ({"solver": "analytic"}, GRAD_PARAMS):
            own = LinearRegression(penalty="l2", alpha=alpha, **solver_params)
            sk = SkRidge(alpha=n_train * alpha)
            row = regression_row(own, sk, data)
            print(
                f"{name:10s} {alpha:5.1f} {solver_params['solver']:>9s} "
                f"{row['coef']:9.2e} {row['intercept']:9.2e} {row['pred']:9.2e}"
                f"   {row['mse_own']:.6f} / {row['mse_sk']:.6f}"
            )
            assert row["coef"] < 1e-8 and row["intercept"] < 1e-8

data = REG_DATA["diabetes"]
wrong = regression_row(
    LinearRegression(penalty="l2", alpha=0.5, solver="analytic"), SkRidge(alpha=0.5), data
)
print(f"diabetes, alpha=0.5, Ridge(alpha=alpha) без множителя n: макс |разность коэффициентов| = {wrong['coef']:.3f}")

## 3. Логистическая регрессия

Сравнение с `sklearn.linear_model.LogisticRegression`. Параметр `penalty` у
`sklearn` устарел (`FutureWarning` с версии 1.8, удаление в 1.10), поэтому
режимы задаются через `C` и `l1_ratio`:

| Режим | Наша модель | `sklearn` |
|---|---|---|
| без штрафа | `penalty=None` | `C=np.inf` |
| L2 | `penalty="l2", alpha=alpha` | `C=1/(n·alpha), l1_ratio=0` |
| L1 | `penalty="l1", alpha=alpha` | `C=1/(n·alpha), l1_ratio=1, solver="saga"` |

Здесь `n` это число объектов в train. Для L1 выбран `saga`: `liblinear`
штрафует свободный член, а у нас и у `saga` он не штрафуется; `lbfgs` L1 не
поддерживает. Параметры нашего спуска: `learning_rate=0.5`, `max_iter=20000`,
`tol=0.0`.

Для проверки точности добавлен независимый эталон: точное решение методом
Ньютона (не использует ни наши классы, ни `sklearn`).

### 3.1. Без штрафа, синтетика

Сравниваются коэффициенты, `intercept`, `predict_proba`, `predict` и accuracy
на test. Отдельно считается расстояние каждой модели до точного решения.

In [ ]:
LOGIT_PARAMS = {"learning_rate": 0.5, "max_iter": 20000, "tol": 0.0}


def newton_reference(X, y, alpha=0.0, max_iter=100):
    """Точное решение (метод Ньютона) для логистической регрессии с L2-штрафом на w.

    Независимый эталон: не использует ни наши классы, ни sklearn.
    Возвращает (coef, intercept). Свободный член не штрафуется.
    """
    n, d = X.shape
    Xa = np.hstack([X, np.ones((n, 1))])
    theta = np.zeros(d + 1)
    R = np.diag([alpha] * d + [0.0])
    for _ in range(max_iter):
        p = 1 / (1 + np.exp(-(Xa @ theta)))
        grad = Xa.T @ (p - y) / n + R @ theta
        hess = (Xa.T * (p * (1 - p))) @ Xa / n + R
        step = np.linalg.solve(hess, grad)
        theta = theta - step
        if np.max(np.abs(step)) < 1e-15:
            break
    return theta[:d], theta[d]


def classification_row(own, sk, data):
    """Обучает обе модели на train и возвращает числа для одной строки сравнения."""
    own.fit(data["X_train"], data["y_train"])
    sk.fit(data["X_train"], data["y_train"])
    X_test, y_test = data["X_test"], data["y_test"]
    pred_own, pred_sk = own.predict(X_test), sk.predict(X_test)
    return {
        "coef": max_abs_diff(own.coef_, sk.coef_.ravel()),
        "intercept": abs(own.intercept_ - sk.intercept_[0]),
        "proba": max_abs_diff(own.predict_proba(X_test), sk.predict_proba(X_test)),
        "agree": np.mean(pred_own == pred_sk),
        "acc_own": np.mean(pred_own == y_test),
        "acc_sk": np.mean(pred_sk == y_test),
    }

In [ ]:
data = CLF_DATA["synthetic"]
own = LogisticRegression(**LOGIT_PARAMS)
sk = SkLogisticRegression(C=np.inf, tol=1e-12, max_iter=10000)
row = classification_row(own, sk, data)
w_ref, b_ref = newton_reference(data["X_train"], data["y_train"])
dist_own = max(max_abs_diff(own.coef_, w_ref), abs(own.intercept_ - b_ref))
dist_sk = max(max_abs_diff(sk.coef_.ravel(), w_ref), abs(sk.intercept_[0] - b_ref))

print("synthetic, без штрафа: свой спуск vs LogisticRegression(C=np.inf)")
print(f"  макс |разность коэффициентов|      : {row['coef']:.2e}")
print(f"  |разность intercept|               : {row['intercept']:.2e}")
print(f"  макс |разность predict_proba|      : {row['proba']:.2e}")
print(f"  доля совпавших классов (predict)   : {row['agree']:.4f}")
print(f"  accuracy на test свой / sklearn    : {row['acc_own']:.4f} / {row['acc_sk']:.4f}")
print(f"  расстояние до точного решения (Ньютон), свой / sklearn: {dist_own:.2e} / {dist_sk:.2e}")
print("  свои coef_:", own.coef_, " intercept_:", round(own.intercept_, 6))
print("  sklearn coef_:", sk.coef_.ravel(), " intercept_:", round(float(sk.intercept_[0]), 6))
print("  n_iter_ свой:", own.n_iter_, " sklearn:", sk.n_iter_)
assert row["coef"] < 1e-6 and row["intercept"] < 1e-6 and row["proba"] < 1e-6
assert row["agree"] == 1.0
assert dist_own < 1e-10